# BlueField-3 DPU: OVS Hardware Offload

The previous notebook (`bluefield_dpus_l2_network_basic_auto.ipynb`) showed how to bring a BlueField-3 DPU
up on FABRIC: push the BFB bundle, reach the ARM cores over the rshim/`tmfifo` link, and give them internet
access. This notebook picks up from there and uses the DPU for the thing it was actually built for —
**moving the packet-forwarding datapath off of general-purpose CPU cores and into the NIC's embedded switch.**

## What hardware offload actually is

A BlueField-3 is two machines in one card:

- a **ConnectX-7 NIC** with an **eSwitch** — an embedded ASIC that can match and forward packets at line rate, and
- an **ARM SoC** running its own Ubuntu, which owns that eSwitch.

In DPU (`switchdev`) mode, the ARM's OS sees a *representor* netdev for every port of the eSwitch:

| Representor | What it represents |
|---|---|
| `p0`, `p1` | the physical uplinks (the wire) |
| `pf0hpf`, `pf1hpf` | the PF that the **host VM** sees |
| `pf0vf0`, … | individual SR-IOV VFs given to the host |

Open vSwitch runs **on the ARM**, bridging `pf0hpf` to `p0`. Every packet the host VM sends therefore crosses the
DPU before it reaches the wire. The interesting question is *where* it gets switched:

- **`hw-offload=false`** — OVS's kernel datapath on the ARM cores touches **every single packet**. Correct, but the
  ARM's cores become the bottleneck, and they are small power-efficient cores, not Xeons.
- **`hw-offload=true`** — OVS still handles the *first* packet of each flow in software, then programs the resulting
  match/action rule into the eSwitch through the Linux **TC flower** interface. Every subsequent packet of that flow
  is switched in hardware and **never reaches a CPU at all**.

That is the whole value proposition of a DPU in one toggle, and it is directly measurable. This notebook measures it.

## What this notebook does

1. Builds a two-node slice: **Node1 carries a BlueField-3**, Node2 is an ordinary peer on a `NIC_Basic`.
2. Installs the BFB bundle and brings up the management path to Node1's DPU.
3. Builds an explicit OVS bridge (`br-doca`) on that DPU: `p0` ↔ `pf0hpf`.
4. Runs an `iperf3` benchmark **host-to-host** with offload **on**, then with offload **off**, recording throughput,
   TCP retransmits, **ARM CPU utilization**, and the number of flows resident in hardware.
5. Compares the two runs side by side.

> **Only one end needs a DPU.** Offload is a property of the *local* eSwitch: when Node1's offload is disabled, its
> ARM cores become the bottleneck for the whole path regardless of what sits at the far end. Keeping Node2 on a plain
> `NIC_Basic` costs nothing experimentally, leaves exactly one variable in the comparison, and does not tie up a
> second BlueField.

> **A note on whose CPU is saved.** Hardware offload relieves the **DPU's ARM cores**, not the host VM's vCPUs — the
> host is doing ordinary sockets-and-driver work either way. So the CPU number this notebook reports is sampled on
> Node1's ARM, over the rshim link. Watching host CPU here would show you very little.

## Topology

```
        FABRIC management network  (this notebook's control channel)
        ──────────┬──────────────────────────────┬──────────────
                  │                              │
          ┌───────┴────────┐             ┌───────┴────────┐
          │  Node1 (VM)    │             │  Node2 (VM)    │
          │  dpu_ubuntu_24 │             │ default_ubuntu │
          └───┬────────────┘             └───┬────────────┘
              │ PCIe passthrough              │ NIC_Basic
     ┌────────┴─────────────┐                 │ (shared vNIC)
     │ BlueField-3          │                 │
     │                      │                 │
     │  pf0hpf ─┐           │                 │
     │          ├─ br-doca  │                 │
     │  p0 ─────┘  (OVS on  │                 │
     │   │         ARM)     │                 │
     └───┼──────────────────┘                 │
         │                                    │
         └──────── L2 network ────────────────┘
                  192.168.1.0/24
```

Node1's traffic is switched by the DPU; Node2's is not. That asymmetry is the point — the only thing that changes
between the two benchmark runs is whether Node1's eSwitch is doing the forwarding.

Two independent paths reach the DPU, and it matters which is which:

- **Control** — this notebook → Node1 over the FABRIC management IP → the DPU over `tmfifo_net0` (`192.168.100.1` ↔
  `192.168.100.2`, an rshim/USB-like channel that does not involve the eSwitch at all).
- **Data** — Node1's PF → `pf0hpf` → `br-doca` → `p0` → the wire → Node2.

Because the two are disjoint, restarting Open vSwitch on the ARM tears down the *experiment's* datapath without ever
touching the notebook's ability to talk to either machine. That is what makes the A/B comparison below safe to run.

## Import the FABlib Library

In [ ]:
import ipaddress
import json
import time
from ipaddress import IPv4Network

import pandas as pd

from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

fablib = fablib_manager()

fablib.show_config();

## Create the Experiment Slice

Node1 gets a `NIC_ConnectX_7_400` BlueField-3 attached by PCI passthrough and runs the `dpu_ubuntu_24` image, which
ships the DOCA stack and the BFB bundle the DPU will be flashed with. Node2 is just a traffic peer, so it takes a
shared `NIC_Basic` and the ordinary `default_ubuntu_24` image.

One port from each card joins an L2 network on a single site, set to `auto` mode so FABlib assigns addresses out of
`192.168.1.0/24` and configures the host-side interfaces during post-boot setup.

### Available BlueField NIC Component Models
- **NIC_ConnectX_7_100**: 100 Gbps Dedicated Mellanox BlueField-3 ConnectX-7 PCI Device (2 Ports)
- **NIC_ConnectX_7_400**: 400 Gbps Dedicated Mellanox BlueField-3 ConnectX-7 PCI Device (2 Ports)

> **`NIC_Basic` is a shared vNIC (an SR-IOV VF), so the far end caps the absolute throughput you can reach.** That is
> fine here: the result worth reading is the *delta* between the two runs and the ARM CPU it costs, not the peak
> Gbps. If you want headroom to see the offloaded path stretch its legs, give Node2 a dedicated NIC too.

In [ ]:
slice_name = 'MySlice-bluefield-offload'

# Both nodes must land on the same site for an L2Bridge.
#site = fablib.get_random_site()
site = "SALT"
print(f"Site: {site}")

node1_name = 'Node1'   # carries the BlueField-3
node2_name = 'Node2'   # ordinary traffic peer
network_name = 'net1'

node1_image = "dpu_ubuntu_24"
node2_image = "default_ubuntu_24"
subnet = IPv4Network("192.168.1.0/24")

In [ ]:
slice = fablib.new_slice(name=slice_name)

net1 = slice.add_l2network(name=network_name, subnet=subnet)

# Node1: BlueField-3. Port 0 of the card carries the experiment traffic; it is
# the port that maps to the `p0` representor on the ARM side.
node1 = slice.add_node(name=node1_name, site=site, image=node1_image)
dpu = node1.add_component(model='NIC_ConnectX_7_400', name='nic1')
iface1 = dpu.get_interfaces()[0]
iface1.set_mode('auto')
net1.add_interface(iface1)

# Node2: plain shared NIC, no DPU.
node2 = slice.add_node(name=node2_name, site=site, image=node2_image)
iface2 = node2.add_component(model='NIC_Basic', name='nic1').get_interfaces()[0]
iface2.set_mode('auto')
net1.add_interface(iface2)

slice.submit();

## Install the BFB Bundle on the DPU

`component.configure()` runs the default command list for the component model, which brings up `tmfifo_net0` on the
host and pushes the BFB image to the DPU over rshim:

```
sudo ip addr add 192.168.100.1/24 dev tmfifo_net0
sudo ip link set tmfifo_net0 up
sudo bfb-install --bfb /opt/bf-bundle/bf-bundle-3.4.0-92_26.04_ubuntu-24.04_64k_prod.bfb --rshim rshim0
```

This reflashes the ARM's OS and takes several minutes. Pass a list of command strings to `configure(commands)` to run
something else instead. Only Node1 is touched here — Node2 has no DPU.

In [ ]:
slice = fablib.get_slice(slice_name)

node1 = slice.get_node(name=node1_name)
node2 = slice.get_node(name=node2_name)

print(f"Installing BFB on {node1.get_name()} ...")
bluefield = node1.get_component(name='nic1')
bluefield.configure()
print("done")

### Post-Install: Reboot and Re-establish the Management Path

Reflashing the DPU resets the host's view of the card, so Node1 is rebooted and the `tmfifo_net0` management address
is reapplied. An SSH key is also generated on Node1 — `bf3_rshim.sh` later refuses to run without one, because it
talks to the DPU key-only.

In [ ]:
node1.execute("sudo reboot")
slice.wait_ssh()
print("Node1 back up")

In [ ]:
node1.execute("sudo ip addr add 192.168.100.1/24 dev tmfifo_net0 || true")
node1.execute("sudo ip link set tmfifo_net0 up")
node1.execute("test -f ~/.ssh/id_ed25519 || ssh-keygen -t ed25519 -f ~/.ssh/id_ed25519 -N ''")

## Manual Step: First Login to the DPU

This one step cannot be automated, because the DPU forces a password change on first login after a BFB install.
Open a terminal on **Node1** and run:

```
ssh ubuntu@192.168.100.2
```

The initial credentials are **`ubuntu` / `ubuntu`**; you will be prompted to choose a new password. **Remember it** —
the next command needs it. Then, still from Node1, install your key so everything below can run unattended:

```
ssh-copy-id -i ~/.ssh/id_ed25519 ubuntu@192.168.100.2
```

If SSH will not come up at all, the DPU console is reachable from the host with `screen /dev/rshim0/console`.

Run the next cell to confirm the DPU answers over key-based SSH before continuing — everything downstream depends
on it.

In [ ]:
DPU_IP = "192.168.100.2"
DPU_USER = "ubuntu"
SSH_OPTS = (
    "-o BatchMode=yes -o StrictHostKeyChecking=no "
    "-o UserKnownHostsFile=/dev/null -o LogLevel=ERROR"
)


def dpu_exec(command, node=None, quiet=True, **kwargs):
    """Run a command on Node1's DPU (ARM cores), by way of the host VM over the rshim link."""
    node = node or node1
    escaped = command.replace("'", "'\"'\"'")
    return node.execute(
        f"ssh {SSH_OPTS} {DPU_USER}@{DPU_IP} '{escaped}'", quiet=quiet, **kwargs
    )


stdout, stderr = dpu_exec("hostname && uname -r")
print(f"DPU -> {stdout.strip()}")

## Give the DPU Internet Access

The ARM cores need to reach the package repositories and, more importantly, a working route out is what
`bf3_rshim.sh` sets up along with the rest of the rshim plumbing. It configures forwarding, NAT and DNS on Node1 so
the DPU can route out through the host's management interface. The mode must match the address family of Node1's
management IP.

In [ ]:
node1.upload_directory("node_tools", ".")

ip = ipaddress.ip_address(node1.get_management_ip())
mode = "ipv4" if ip.version == 4 else "ipv6"
print(f"Node1 management IP is {mode}, configuring NAT ...")

node1.execute(f"sudo ./node_tools/bf3_rshim.sh --mode {mode}")

In [ ]:
# Confirm the DPU can actually resolve and reach the outside world.
stdout, stderr = dpu_exec("getent hosts archive.ubuntu.com || echo 'DNS FAILED'")
print(f"DPU DNS -> {stdout.strip()}")

## Inspect the eSwitch and Its Representors

Before changing anything, look at what the BFB install left behind. Three things are worth confirming:

- **`devlink dev eswitch show`** should report `mode switchdev`. In the alternative `legacy` mode there are no
  representors and no offload — if you see `legacy` here, the card is not in DPU mode and nothing below will work.
- **`ip -br link`** should list `p0`/`p1` (uplinks) and `pf0hpf`/`pf1hpf` (host PF representors).
- **`ovs-vsctl show`** will already show bridges: recent BFB images ship `ovsbr1` (`p0` + `pf0hpf`) and `ovsbr2`
  (`p1` + `pf1hpf`) preconfigured, usually with `hw-offload` already enabled.

That last point is why the next section rebuilds the bridge explicitly rather than assuming a clean slate.

In [ ]:
stdout, _ = dpu_exec(
    "for d in $(sudo devlink dev show | awk '{print $1}'); do "
    "printf '%s: ' $d; sudo devlink dev eswitch show $d 2>/dev/null || echo '(no eswitch)'; done"
)
print(f"eswitch mode :\n{stdout}")

stdout, _ = dpu_exec("ip -br link show | grep -E '^(p[0-9]|pf[0-9]hpf)' || true")
print(f"representors :\n{stdout}")

stdout, _ = dpu_exec("sudo ovs-vsctl show")
print(f"existing OVS :\n{stdout}")

stdout, _ = dpu_exec("sudo ovs-vsctl get Open_vSwitch . other_config")
print(f"other_config : {stdout.strip()}")

## Build the Offload Bridge

The preinstalled bridges are replaced with a single explicit one so the experiment's datapath is unambiguous:

```
pf0hpf  ──┐
          ├── br-doca
p0 ───────┘
```

`pf0hpf` is Node1's traffic arriving at the DPU; `p0` is the wire. Bridging them is what makes Node1 reachable at
all — **with no bridge, the host VM has no connectivity through this port**, which is worth internalizing: on a DPU,
the ARM is not an optional accessory sitting off to the side, it *is* the datapath.

`vswitchd` is restarted afterwards so the bridge is rebuilt from a known state.

In [ ]:
dpu_exec("sudo ovs-vsctl --if-exists del-br ovsbr1")
dpu_exec("sudo ovs-vsctl --if-exists del-br ovsbr2")
dpu_exec("sudo ovs-vsctl --if-exists del-br br-doca")

dpu_exec("sudo ovs-vsctl add-br br-doca")
dpu_exec("sudo ovs-vsctl add-port br-doca p0")
dpu_exec("sudo ovs-vsctl add-port br-doca pf0hpf")

for link in ["p0", "pf0hpf", "br-doca"]:
    dpu_exec(f"sudo ip link set dev {link} up")

stdout, _ = dpu_exec("sudo ovs-vsctl show")
print(stdout)

## Install the Measurement Tools

`iperf3` runs on the **host VMs** (it generates the traffic that crosses the DPU). The ARM CPU sample is taken
straight from `/proc/stat`, so nothing extra is needed on the DPU itself.

In [ ]:
for node in [node1, node2]:
    node.execute(
        "sudo apt-get update -qq && sudo DEBIAN_FRONTEND=noninteractive "
        "apt-get install -y -qq iperf3 > /dev/null",
        quiet=True,
    )
    stdout, _ = node.execute("iperf3 --version | head -1")
    print(f"{node.get_name()}: {stdout.strip()}")

## Verify the Data Path

Before benchmarking, confirm the two hosts can reach each other — from Node1 that means through the DPU. The
addresses come from the `auto` mode assignment FABlib made out of `192.168.1.0/24`.

If this ping fails, re-run `node1.config()` and bring the interfaces up by hand — the VM's OS does not always notice
the interfaces after the DPU is reflashed.

In [ ]:
node1_addr = node1.get_interface(network_name=network_name).get_ip_addr()
node2_addr = node2.get_interface(network_name=network_name).get_ip_addr()
print(f"{node1_name}: {node1_addr}")
print(f"{node2_name}: {node2_addr}")

stdout, stderr = node1.execute(f"ping -c 5 {node2_addr}")

## The Benchmark

Each run does the same thing:

1. Set `hw-offload` on the DPU and restart `vswitchd`, then let the datapath settle.
2. Sample `/proc/stat` on the ARM.
3. Run `iperf3` from Node1 to Node2 for `DURATION` seconds across `STREAMS` parallel TCP streams, in JSON mode.
4. Sample `/proc/stat` again, and count how many flows are resident in hardware versus software.

`dpctl/dump-flows type=offloaded` lists the flows the eSwitch is switching on its own; `type=non-offloaded` lists
the ones still being handled by the ARM's kernel datapath. With offload enabled the first number should be
non-zero and the second near zero, and with it disabled the reverse.

Note that flow counts are sampled *after* the run, and OVS ages idle flows out after ~10 s — so read them as
"was the hardware being used", not as a precise census.

> **FABlib tip:** the `iperf3` server is started with **`node.execute_thread()`**, not `node.execute()`.
> `execute()` blocks until the remote command closes its output streams, and a server holds them open for as long
> as it runs — `iperf3 -s -D` does not help, because the daemon inherits those streams and the call hangs anyway.
> `execute_thread()` hands back a future instead, so the notebook can go on to drive the client. Pair it with
> `iperf3 -s -1` and the server retires itself after one test.

In [ ]:
DURATION = 20
STREAMS = 4


def cpu_sample():
    """Aggregate jiffies from the DPU's /proc/stat as (total, idle)."""
    stdout, _ = dpu_exec("grep '^cpu ' /proc/stat")
    fields = [int(x) for x in stdout.split()[1:]]
    return sum(fields), fields[3] + fields[4]  # idle + iowait


def cpu_busy_pct(before, after):
    d_total = after[0] - before[0]
    d_idle = after[1] - before[1]
    return 100.0 * (d_total - d_idle) / d_total if d_total else float("nan")


def set_hw_offload(enabled, settle=25):
    value = "true" if enabled else "false"
    dpu_exec(f"sudo ovs-vsctl set Open_vSwitch . other_config:hw-offload={value}")
    dpu_exec("sudo systemctl restart openvswitch-switch")
    time.sleep(settle)
    for link in ["p0", "pf0hpf", "br-doca"]:
        dpu_exec(f"sudo ip link set dev {link} up")


def flow_counts():
    offloaded, _ = dpu_exec("sudo ovs-appctl dpctl/dump-flows type=offloaded | wc -l")
    software, _ = dpu_exec("sudo ovs-appctl dpctl/dump-flows type=non-offloaded | wc -l")
    return int(offloaded.strip() or 0), int(software.strip() or 0)


def run_benchmark(hw_offload):
    label = "offload ON" if hw_offload else "offload OFF"
    print(f"===== {label} =====")

    set_hw_offload(hw_offload)

    # Warm the path so the very first packets' flow-setup cost is not counted.
    node1.execute(f"ping -c 3 {node2_addr}", quiet=True)

    node2.execute("pkill -f 'iperf3 -s' || true", quiet=True)

    # `execute` blocks until the remote command closes stdout/stderr, and an iperf3
    # server holds them open for as long as it runs -- `-D` does not help, because
    # the daemon inherits them. Run the server in its own thread instead, with `-1`
    # so it retires after this one test.
    server = node2.execute_thread("iperf3 -s -1")
    time.sleep(2)

    before = cpu_sample()

    stdout, stderr = node1.execute(
        f"iperf3 -c {node2_addr} -t {DURATION} -P {STREAMS} -J", quiet=True
    )

    after = cpu_sample()

    node2.execute("pkill -f 'iperf3 -s' || true", quiet=True)
    server.result()  # the thread returns once the server process is gone

    result = json.loads(stdout)
    offloaded, software = flow_counts()

    row = {
        "configuration": label,
        "throughput (Gbps)": round(
            result["end"]["sum_received"]["bits_per_second"] / 1e9, 2
        ),
        "retransmits": result["end"]["sum_sent"].get("retransmits"),
        "DPU ARM CPU (%)": round(cpu_busy_pct(before, after), 1),
        "hw flows": offloaded,
        "sw flows": software,
    }
    print(row)
    return row

### Run 1 — Hardware Offload Enabled

This is how a DPU is meant to be run. OVS programs the eSwitch through TC flower, and the ARM cores see only the
first packet of each flow.

In [ ]:
results_on = run_benchmark(hw_offload=True)

In [ ]:
# The flows the eSwitch is switching without CPU involvement.
stdout, _ = dpu_exec("sudo ovs-appctl dpctl/dump-flows type=offloaded | head -20")
print(stdout)

# The same rules, seen through the kernel's TC layer -- note 'in_hw'.
stdout, _ = dpu_exec("sudo tc -s filter show dev p0 ingress | head -40")
print(stdout)

### Run 2 — Hardware Offload Disabled

Now the same traffic with the eSwitch taken out of the loop: every packet is copied up to the ARM's kernel
datapath, switched in software, and copied back down. Nothing about the topology changes — only where the work
happens.

In [ ]:
results_off = run_benchmark(hw_offload=False)

In [ ]:
# Expect this to be empty now: no rules are being pushed to hardware.
stdout, _ = dpu_exec("sudo ovs-appctl dpctl/dump-flows type=offloaded | head -20")
print(f"offloaded flows:\n{stdout}")

stdout, _ = dpu_exec("sudo ovs-appctl dpctl/dump-flows type=non-offloaded | head -20")
print(f"software flows:\n{stdout}")

## Compare

In [ ]:
comparison = pd.DataFrame([results_on, results_off]).set_index("configuration")
comparison

In [ ]:
gbps_on = results_on["throughput (Gbps)"]
gbps_off = results_off["throughput (Gbps)"]
cpu_on = results_on["DPU ARM CPU (%)"]
cpu_off = results_off["DPU ARM CPU (%)"]

speedup = f"{gbps_on / gbps_off:.1f}x" if gbps_off else "n/a"
print(f"Throughput   : {gbps_off:.2f} -> {gbps_on:.2f} Gbps  ({speedup})")
print(f"ARM CPU busy : {cpu_off:.1f}% -> {cpu_on:.1f}%")

## Reading the Results

**Throughput.** The software datapath is limited by how fast the ARM cores can push packets, so the offloaded run
should be faster — and the gap widens as you add streams or shrink the MTU, because the software path is bounded by
*packets per second* while the hardware path is bounded by the wire. Remember that Node2's `NIC_Basic` is a shared
vNIC and puts a ceiling on the absolute numbers; if the two runs come out close and the ARM was never busy, you are
measuring that ceiling rather than the DPU.

**ARM CPU.** The number to watch, and the one the far end cannot distort. With offload on, the ARM is nearly idle
during a run: it set up a handful of flows and then had nothing to do. With offload off, it climbs steeply. This is
the difference between a DPU that has capacity left over to run your workload and one entirely consumed by moving
packets.

**Flow counts.** `hw flows` non-zero with offload on is the direct confirmation that rules reached the eSwitch. The
`in_hw` marker in the `tc filter show` output above says the same thing at the kernel level.

**If the two runs look identical**, offload was probably never active in the first run. Check, in order: that
`devlink dev eswitch show` reported `switchdev`; that `hw flows` was non-zero; and `dmesg` on the ARM for TC
rejections — an unsupported match or action makes OVS silently fall back to software for that flow.

### Things worth trying next

- Scale `STREAMS` up. Software forwarding degrades with flow count in a way hardware forwarding does not.
- Give Node2 a dedicated NIC (or a second BlueField) to lift the far-end ceiling and let the offloaded path stretch.
- Add VLAN tags or an OVS tunnel (VXLAN/GENEVE) to the bridge. Encap/decap is offloadable on ConnectX-7, and
  watching a tunneled flow land `in_hw` is a much stronger result than plain L2 forwarding.
- Install an OpenFlow rule that rewrites or drops a subset of the traffic, then check whether it stayed in hardware.
  Finding the edge where offload stops working teaches you more about the card than any benchmark does.

## Restore and Clean Up

Re-enable offload so the slice is left in the sane configuration, then delete the slice when you are finished.

In [ ]:
set_hw_offload(True)
stdout, _ = dpu_exec("sudo ovs-vsctl get Open_vSwitch . other_config")
print(f"DPU other_config: {stdout.strip()}")

## Delete the Slice

Please delete your slice when you are done with your experiment.

In [ ]:
slice = fablib.get_slice(slice_name)
slice.delete()